<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/sweep_ckpt_circuit_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

In [2]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "sojup/entity_binding_test"
FILENAME = "D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [ ]:
REPO_ID = "sojup/entity_binding_test"
FILENAME = "id_mapping.csv"

id_mapping_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [6]:
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

# SEP = E + T
# Q = E + T + 1
# PAD = E + T + 2
D_VOCAB = E + T + 3

# ENTITIES = np.arange(0, E)
# TYPES    = np.arange(E, E + T)

# N_WORLDS = 80_000
# MIN_FACTS, MAX_FACTS = 4, 8
# SEED = 0

# rng = np.random.default_rng(SEED)

# def produce_example(num_relations: int, *, allow_self_loops: bool = False):
#     facts = []
#     seen_head_rel = set()
#     seen_e = set()
#     seen_t = set()

#     while len(facts) < num_relations:
#         e = int(rng.integers(0, E))
#         t = int(TYPES[rng.integers(0, T)])

#         # enforce uniqueness of e and t (not just the tuple)
#         if e in seen_e or t in seen_t:
#             continue
#         if (e, t) in seen_head_rel:
#             continue

#         # forbid self-loop (optional)
#         e2 = int(rng.integers(0, E))
#         while not allow_self_loops and e2 == e:
#             e2 = int(rng.integers(0, E))

#         seen_head_rel.add((e, t))
#         seen_e.add(e)
#         seen_t.add(t)
#         facts.append((e, t, e2))

#     q_idx = int(rng.integers(0, num_relations))
#     Eq, Tq, E2q = facts[q_idx]

#     seq = []
#     for (e, t, e2) in facts:
#         seq.extend([e, t, e2, SEP])
#     seq.extend([Tq, Eq, Q])

#     return seq, E2q, facts



# rows = []
# for _ in tqdm(range(N_WORLDS)):
#     k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
#     seq, label, _ = produce_example(k, allow_self_loops=False)
#     rows.append({"tokens": seq, "label": label})

# df = pd.DataFrame(rows)

In [7]:
N_LAYERS = 3
HEADS = 2

d_model = 256
d_mlp   = 1024
n_ctx   = 64
lr = 3e-4
betas = (0.9, 0.98)
weight_decay = 0.01
num_epochs = 30

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        d_mlp=d_mlp,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        act_fn="gelu",
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

Moving model to device:  cuda
Model loaded successfully.


In [8]:
id_mapping_df = pd.read_csv(id_mapping_path)
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [9]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [ ]:
from datasets import load_dataset

dataset = load_dataset("sojup/entity_binding", split="test")

In [14]:
test_df = dataset.to_pandas()
test_dataset = EntityBindingDataset(test_df)

In [15]:
### Some very basic checks: What happens if we pertube the input sequence

In [16]:
id_to_entity[210] = ","
id_to_entity[211] = "?"
example, label = test_dataset[0]

In [17]:
print("Example tokens and their mapped entities:")
for token_id in example:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example tokens and their mapped entities:
Token 10 (Jeffery)
Token 104 (Mumbai)
Token 1 (Angel)
Token 110 (Kolkata)
Token 20 (Linda)
Token 103 (Guangzhou)
Token 30 (Lindsay)
Token 110 (Kolkata)
Token 70 (Jason)
Token 105 (Manila)
Token 37 (Christine)
Token 110 (Kolkata)
Token 30 (Lindsay)
Token 100 (Tokyo)
Token 24 (Susan)
Token 110 (Kolkata)
Token 105 (Manila)
Token 70 (Jason)
Token 111 (Sao Paulo)

Label: 37 (Christine)


In [18]:
## considering the incorrect token to be the 10th most probable prediction
## take a batch of examples with this heuristic

In [19]:
try:
    import google.colab # type: ignore
    IN_COLAB = True
except:
    IN_COLAB = False

import os, sys
chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"

if IN_COLAB:
    # Install packages
    %pip install transformer_lens
    %pip install einops
    %pip install jaxtyping
    %pip install git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

    # Code to download the necessary files (e.g. solutions, test funcs)
    if not os.path.exists(f"/content/{chapter}"):
        !wget https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/main.zip
        !unzip /content/main.zip 'ARENA_3.0-main/chapter1_transformer_interp/exercises/*'
        sys.path.append(f"/content/{repo}-main/{chapter}/exercises")
        os.remove("/content/main.zip")
        os.rename(f"{repo}-main/{chapter}", chapter)
        os.rmdir(f"{repo}-main")
        os.chdir(f"{chapter}/exercises")
else:
    chapter_dir = r"./" if chapter in os.listdir() else os.getcwd().split(chapter)[0]
    sys.path.append(chapter_dir + f"{chapter}/exercises")

  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-r7bs7kiv
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-r7bs7kiv
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for circuitsvis: filename=circuitsvis-0.0.0-py3-none-any.whl size=6172337 sha256=a4f68a36d13a4fbe2ed1d8bde97d80050bd39c247f48ad5a9f309c1c97601ffd
  Stored in directory: /tmp/pip-ephem-wheel-cache-fhgnk2la/wheels/00/ce/19/651aed367fa8cefad943dece40a2248cef6588697047472ef1
Successfully built circuitsvis
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 8.7.0
    Uninstalling importlib_metadata-8.7.0:
      Successfully uninstalled importlib_metadata-8.7.0
--2025-

## Direct Logit Attribution/Logit Lens

In [20]:
from rich.table import Table, Column, box
from rich import print as rprint
from jaxtyping import Float, Int, Bool
from typing import List, Optional, Callable, Tuple, Dict, Literal, Set, Union
from torch import Tensor
from transformer_lens import ActivationCache
import einops
from rich import print as rprint
from plotly_utils import imshow, line, scatter, bar
from pathlib import Path
from IPython.display import display, HTML
from transformer_lens import utils
import circuitsvis as cv


In [21]:
import random
random.seed(42)
same_seq_subset = [ex for ex in test_dataset if len(ex[0]) == 23]

start, end = 0, len(same_seq_subset)
count = 10
random_ints = [random.randint(start, end) for _ in range(count)]

dla_dataset = [same_seq_subset[idx] for idx in random_ints]


In [22]:
text_examples = []
text_labels = []
text_incorrect_answers = []

incorrect_tokens = []
label_tokens = []
example_tokens = []

top1_probs = []
top10_probs = []

logit_diffs = []
acc = 0

for ex in dla_dataset:
  # get model prediction
  input, label = ex
  with torch.no_grad():
    logits = model(input)

  probs = logits[0, -1, :].softmax(dim=-1)
  top_values, top_indices = probs.topk(10)
  incorrect_token = top_indices.tolist()[-1]

  text_incorrect_answers.append(id_to_entity[incorrect_token])
  incorrect_tokens.append(incorrect_token)

  text_examples.append(" ".join([id_to_entity[tok] for tok in ex[0].tolist()]))
  example_tokens.append(ex)

  text_labels.append(id_to_entity[label.item()])
  label_tokens.append(label.item())

  top1_probs.append(top_values.tolist()[0])
  top10_probs.append(top_values.tolist()[-1])

  correct_token = top_indices.tolist()[0]
  if correct_token == label.item():
    acc += 1

  logit_diffs.append(logits[0, -1, label.item()] - logits[0, -1, incorrect_token])

In [23]:
cols = [
    "Prompt",
    Column("Correct", style="rgb(0,200,0) bold"),
    Column("Incorrect", style="rgb(255,0,0) bold"),
    Column("Logit Difference", style="bold")
]
table = Table(*cols, title="Logit differences", show_lines=True)

for prompt, answer, incorrect_answer, logit_diff in zip(text_examples, text_labels, text_incorrect_answers, logit_diffs):
    table.add_row(prompt, repr(answer), repr(incorrect_answer), f"{logit_diff.item():.3f}")

rprint(table)

                                                 Logit differences                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Prompt                                                            ┃ Correct    ┃ Incorrect   ┃ Logit Difference ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Brandon Seoul Jill Kolkata Lisa Shanghai Amanda Kolkata Angel     │ 'Jill'     │ 'Jeffery'   │ 15.381           │
│ Mexico City Sharon Kolkata Andrea Cairo Bryan Kolkata Anthony     │            │             │                  │
│ Jakarta Noah Kolkata Seoul Brandon Sao Paulo                      │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Danielle Delhi Susan Kolkata Robert Mexico City Wendy Kolkata     │ 'Jason'    │ 'Adam'      │ 17.180           │
│ Teresa Jakarta Colin Kolkata Nicholas Manila Jason Kolkata        │            │             │                  │
│ Christopher Mumbai Erica Kolkata Manila Nicholas Sao Paulo        │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Jeremy Seoul Jeremy Kolkata Helen Jakarta Sharon Kolkata Amy      │ 'Jeremy'   │ 'Mitchell'  │ 15.573           │
│ Shanghai George Kolkata Leslie Cairo Jason Kolkata Adam Mumbai    │            │             │                  │
│ Anthony Kolkata Seoul Jeremy Sao Paulo                            │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Frank Seoul Robin Kolkata Jill Jakarta Adam Kolkata Angel         │ 'Jose'     │ 'Randall'   │ 18.212           │
│ Shanghai Mitchell Kolkata Erica Mexico City Anthony Kolkata       │            │             │                  │
│ Brenda Tokyo Jose Kolkata Tokyo Brenda Sao Paulo                  │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica  │ 'Jeremy'   │ 'Cassandra' │ 21.483           │
│ Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney │            │             │                  │
│ Cairo Sherri Kolkata Manila Jessica Sao Paulo                     │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Tasha Jakarta William Kolkata John Tokyo Christopher Kolkata      │ 'Linda'    │ 'Maria'     │ 24.731           │
│ Andrea Delhi Joseph Kolkata Amanda Manila Joseph Kolkata Keith    │            │             │                  │
│ Seoul Linda Kolkata Seoul Keith Sao Paulo                         │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Patricia Seoul Courtney Kolkata John Mumbai Debra Kolkata Laura   │ 'Matthew'  │ 'Rachael'   │ 19.041           │
│ Tokyo Jesse Kolkata Christine Cairo Anna Kolkata Robert Guangzhou │            │             │                  │
│ Matthew Kolkata Guangzhou Robert Sao Paulo                        │            │             │                  │
├───────────────────────────────────────────────────────────────────┼────────────┼─────────────┼──────────────────┤
│ Brandon Mumbai Reginald Kolkata Joseph Seoul Jeffrey Kolkata      │ 'Reginald' │ 'Rachael'   │ 16.414           │
│ Robert Guangzhou Reginald Kolkata Lisa Jakarta Leslie Kolkata     │            │             │                  │
│ Jose Shanghai Anthony Kolkata Mumbai Brandon Sao Paulo

In [24]:
## logit lens

In [25]:
def residual_stack_to_logit_diff(
    residual_stack: Float[Tensor, "... batch d_model"], # contains residual stream values for the final sequence position
    cache: ActivationCache,
    logit_diff_directions: Float[Tensor, "batch d_model"],
) -> Float[Tensor, "..."]:
    '''
    Gets the avg logit difference between the correct and incorrect answer for a given
    stack of components in the residual stream.
    '''
    print(residual_stack.shape)
    ln_residual_stack = cache.apply_ln_to_stack(residual_stack, layer=-1, pos_slice=-1)
    average_logit_diff = einops.einsum(ln_residual_stack, logit_diff_directions, "... batch d_model, batch d_model ->...") / residual_stack.shape[0]
    return average_logit_diff

In [26]:
clean_dataset = torch.stack([ex[0] for ex in dla_dataset], dim=0)
clean_dataset.shape

torch.Size([10, 23])

In [27]:
# this is equivalent to indexing the unembedding matrix and getting the column corresponding to the given index/token
answer_token_directions = model.tokens_to_residual_directions(torch.tensor(label_tokens))
incorrect_token_directions = model.tokens_to_residual_directions(torch.tensor(incorrect_tokens))
logit_diff_directions = answer_token_directions - incorrect_token_directions


In [28]:
# verification of the method
(answer_token_directions[0] == model.W_U[:, label_tokens[0]]).sum() == 256

tensor(True, device='cuda:0')

In [29]:
def make_WV_identity(layer):
  model.blocks[layer].attn.W_V.data.fill_(1.0)

def make_WO_identity(layer):
  model.blocks[layer].attn.W_O.data.fill_(1.0)

In [30]:
original_logits, cache = model.run_with_cache(clean_dataset)

In [31]:
accumulated_residual, labels = cache.accumulated_resid(layer=-1, pos_slice=-1, return_labels=True)
# accumulated_residual has shape (component, batch, d_model)
# 12 blocks, so 12 attn, 12 mlp and one input layer
print("acc", accumulated_residual.shape)
logit_lens_logit_diffs: Float[Tensor, "component"] = residual_stack_to_logit_diff(accumulated_residual, cache, logit_diff_directions)

line(
    logit_lens_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Accumulated Residual Stream",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800
)

acc torch.Size([4, 10, 256])
torch.Size([4, 10, 256])


In [32]:
per_layer_residual, labels = cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)
per_layer_logit_diffs = residual_stack_to_logit_diff(per_layer_residual, cache, logit_diff_directions)

line(
    per_layer_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Each Layer",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800
)

torch.Size([5, 10, 256])


In [33]:
per_head_residual, labels = cache.stack_head_results(layer=-1, pos_slice=-1, return_labels=True)
per_head_residual = einops.rearrange(
    per_head_residual,
    "(layer head) ... -> layer head ...",
    layer=model.cfg.n_layers
)
per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, cache, logit_diff_directions)

imshow(
    per_head_logit_diffs,
    labels={"x":"Head", "y":"Layer"},
    title="Logit Difference From Each Head",
    width=600
)

Tried to stack head results when they weren't cached. Computing head results now
torch.Size([3, 2, 10, 256])


## Attention Analysis

In [34]:
def topk_of_Nd_tensor(tensor: Float[Tensor, "rows cols"], k: int):
    '''
    Helper function: does same as tensor.topk(k).indices, but works over 2D tensors.
    Returns a list of indices, i.e. shape [k, tensor.ndim].

    Example: if tensor is 2D array of values for each head in each layer, this will
    return a list of heads.
    '''
    i = torch.topk(tensor.flatten(), k).indices
    return np.array(np.unravel_index(utils.to_numpy(i), tensor.shape)).T.tolist()


k = 2

for head_type in ["Positive", "Negative"]:

    # Get the heads with largest (or smallest) contribution to the logit difference
    top_heads = topk_of_Nd_tensor(per_head_logit_diffs * (1 if head_type=="Positive" else -1), k)

    # Get all their attention patterns
    attn_patterns_for_important_heads: Float[Tensor, "head q k"] = torch.stack([
        cache["pattern", layer][:, head].mean(0)
         for layer, head in top_heads
    ])

    # Display results
    display(HTML(f"<h2>Top {k} {head_type} Logit Attribution Heads</h2>"))
    display(cv.attention.attention_patterns(
        attention = attn_patterns_for_important_heads,
        tokens = [id_to_entity[tok] for tok in clean_dataset[6].tolist()],
        attention_head_names = [f"{layer}.{head}" for layer, head in top_heads],
    ))

## Analyzing Corruption Techniques

In [35]:
clean = clean_dataset[4]
clean, " ".join([id_to_entity[tok] for tok in clean.tolist()])

(tensor([ 71, 106,  72, 110,  21, 102,  93, 110,  57, 105,  66, 110,  29, 101,
          50, 110,  91, 108,  71, 110, 105,  57, 111]),
 'Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney Cairo Sherri Kolkata Manila Jessica Sao Paulo')

In [36]:
example_tokens[0], text_examples[0]

((tensor([ 89, 107,   4, 110,  18, 106,  32, 110,   1, 109,  53, 110,  35, 108,
           94, 110,   9, 101,  59, 110, 107,  89, 111]),
  tensor(4)),
 'Brandon Seoul Jill Kolkata Lisa Shanghai Amanda Kolkata Angel Mexico City Sharon Kolkata Andrea Cairo Bryan Kolkata Anthony Jakarta Noah Kolkata Seoul Brandon Sao Paulo')

In [37]:
corrupt_entity = 29 # Cassandra
corrupt_relation = 202 # works in
ic_relation = 200 # lives in
ic_entity = 18 # Lisa
ic_entity2 = 28 # Melanie

In [38]:
## clean
print(" ".join([id_to_entity[tok] for tok in clean.tolist()]))
## both entity and relation are not in context
corrupt_no_entity_no_rel = clean.tolist()[:-3] + [corrupt_relation, corrupt_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_entity_no_rel]))
## entity is there but relation not in context
corrupt_no_rel = clean.tolist()[:-3] + [corrupt_relation, ic_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_rel]))
## relation is there but entity not in context
corrupt_no_entity = clean.tolist()[:-3] + [ic_relation, corrupt_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_entity]))
## both entity and relaion are in context, but not binded
corrupt_both_in_context = clean.tolist()[:-3] + [203, ic_entity2, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_both_in_context]))

Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney Cairo Sherri Kolkata Manila Jessica Sao Paulo
Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney Cairo Sherri Kolkata works in Cassandra ?
Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney Cairo Sherri Kolkata works in Lisa ?
Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney Cairo Sherri Kolkata lives in Cassandra ?
Sherri Shanghai Nancy Kolkata Helen Delhi Robert Kolkata Jessica Manila Jeremy Kolkata Cassandra Jakarta Mitchell Kolkata Courtney Cairo Sherri Kolkata travels to Melanie ?


In [39]:
def print_stats(logits):
  probs = logits[0, -1, :].softmax(dim=-1)*100
  top_values, top_indices = probs.topk(10)
  top_values, top_indices = top_values.tolist(), top_indices.tolist()
  for idx in range(10):
    print(f"{top_indices[idx]}, {id_to_entity[top_indices[idx]]}, {top_values[idx]}%")

In [40]:
clean_logits = model(clean)
corrupt1_logits = model(torch.tensor(corrupt_no_entity_no_rel))
corrupt2_logits = model(torch.tensor(corrupt_no_rel))
corrupt3_logits = model(torch.tensor(corrupt_no_entity))
corrupt4_logits = model(torch.tensor(corrupt_both_in_context))

RuntimeError: CUDA error: CUBLAS_STATUS_EXECUTION_FAILED when calling cublasLtMatmul with transpose_mat1 1 transpose_mat2 0 m 256 n 23 k 256 mat1_ld 256 mat2_ld 256 result_ld 256 abcType 0 computeType 68 scaleType 0

In [ ]:
print("Clean")
print_stats(clean_logits)
print("=================================================")
print("No entity no relation in context")
print_stats(corrupt1_logits)
print("==================================================")
print("No relation in context")
print_stats(corrupt2_logits)
print("==================================================")
print("No entity in context")
print_stats(corrupt3_logits)
print("==================================================")
print("No entity entity_type binding in context")
print_stats(corrupt4_logits)

## Activation Patching

In [ ]:
def create_corrupt_example(clean_example):
  """
    Corruption process:
      1. Splits the input into parts using token `210` as a separator.
      2. Identifies the query part (last part in the split).
      3. Finds all context parts except the one that exactly matches the query's
        entity–relation pair (to avoid trivial self-copy).
      4. Randomly selects one part to supply a new entity and another to supply
        a new relation.
      5. Replaces the query's original relation and entity with the randomly
        selected ones.
      6. Returns the corrupted token sequence as a new tensor.

  """
  parts = " ".join(str(tok) for tok in clean_example.tolist()).split("210")

  query_part = parts[-1]
  all_idx = list(range(len(parts)-1))
  for idx, part in enumerate(parts[:-1]):
    if query_part.strip()[:-4].split()[::-1] == part.split()[:-1]:
      all_idx.remove(idx)
      break

  entity_idx = random.choice(all_idx)
  all_idx.remove(entity_idx)
  relation_idx = random.choice(all_idx)
  corrupt_entity_part = parts[entity_idx].strip()
  corrupt_relation_part = parts[relation_idx].strip()
  corrupt_relation = int(corrupt_relation_part.split()[1])
  corrupt_entity = int(corrupt_entity_part.split()[0])
  corrupt_example = clean_example.tolist()[:-3] + [corrupt_relation, corrupt_entity, 211]

  return torch.tensor(corrupt_example)



In [ ]:
corrupt_dataset = []
for clean_example in clean_dataset:
  corrupt_dataset.append(create_corrupt_example(clean_example))

corrupt_dataset = torch.stack(corrupt_dataset)

In [ ]:
def render_clean_and_corrupt(clean_dataset, corrupt_dataset):
  for idx in range(len(clean_dataset)):
    print(f"Clean: {' '.join([id_to_entity[tok] for tok in clean_dataset.tolist()[idx]])}")
    print(f"Corrupt: {' '.join([id_to_entity[tok] for tok in corrupt_dataset.tolist()[idx]])}")
    print("=============================================================================")

In [ ]:
render_clean_and_corrupt(clean_dataset, corrupt_dataset)

In [ ]:
## In IOI we have a clear notion of a correct answer and an incorrect answer given an input.
## Meaning, if we flip/perturb some part of input the output flips in a deterministic manner.
## That is not the case here. Our perturbation is not principled as in IOI and is not guaranteed
## to flip the output.

## current way of getting the incorrect answers, taking the model's prediction on the corrupt dataset

logits = model(corrupt_dataset)
incorrect_answers = logits[:, -1].argmax(dim=-1)
answer_tokens = torch.stack([torch.tensor(label_tokens), incorrect_answers], dim=1)

## the way we calculated incorrect tokens for DLA - take the 10th most probable prediction
## on the clean dataset as the incorrect answer. Usually the probability of this token is super low.

# answer_tokens = torch.stack([torch.tensor(label_tokens), torch.tensor(incorrect_tokens)], dim=1)

In [ ]:
def logits_to_ave_logit_diff(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Float[Tensor, "batch 2"]=answer_tokens,
    per_prompt: bool = False
) -> Union[Float[Tensor, ""], Float[Tensor, "batch"]]:
    '''
    Returns logit difference between the correct and incorrect answer.

    If per_prompt=True, return the array of differences rather than the average.
    '''
    diffs = []
    batch_size, seq = logits.shape[0], logits.shape[1]
    for i in range(batch_size):
      correct_idx = answer_tokens[i,0]
      incorrect_idx = answer_tokens[i,1]
      diff = logits[i, seq-1, correct_idx] - logits[i, seq-1, incorrect_idx]
      diffs.append(diff)
    if per_prompt:
      return torch.FloatTensor(diffs)
    else:
      return torch.FloatTensor(diffs).mean()

In [ ]:
clean_logits, clean_cache = model.run_with_cache(clean_dataset)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupt_dataset)

clean_logit_diff = logits_to_ave_logit_diff(clean_logits, answer_tokens)
print(f"Clean logit diff: {clean_logit_diff:.4f}")

corrupted_logit_diff = logits_to_ave_logit_diff(corrupted_logits, answer_tokens)
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

In [ ]:
def patching_metric(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Float[Tensor, "batch 2"]=answer_tokens,
    corrupted_logit_diff: float=corrupted_logit_diff,
    clean_logit_diff: float=clean_logit_diff,
) -> Float[Tensor, ""]:
    '''
    Linear function of logit diff, calibrated so that it equals 0 when performance is
    same as on corrupted input, and 1 when performance is same as on clean input.
    '''
    logit_diff = logits_to_ave_logit_diff(logits)
    patching_metric = logit_diff/ (clean_logit_diff + -1*corrupted_logit_diff) + 0.5
    patching_metric = torch.FloatTensor(patching_metric)
    return patching_metric

In [ ]:
from transformer_lens import patching
## patching resid_pre
# Run the model with corrupted tokens at each position to get corrupted activations. We then replace these
# corrupted activations with activations from clean cache at each sequence position, to see which component/layer
# and which position improves the performance most.
act_patch_resid_pre = patching.get_act_patch_resid_pre(
    model = model,
    corrupted_tokens = corrupt_dataset, # corrupted input sentences, with S2 swapped with IO. Are we patching with corrupted tokens?
    clean_cache = clean_cache, # clean cache, so clean activations
    patching_metric = patching_metric
)

In [ ]:
labels = [f"{id_to_entity[tok]} {i}" for i, tok in enumerate(clean_dataset[0].tolist())]
imshow(
    act_patch_resid_pre,
    labels={"x": "Position", "y": "Layer"},
    x=labels,
    title="resid_pre Activation Patching",
    width=600
)

In [ ]:
## patch attn_out
act_patch_attn_out = patching.get_act_patch_attn_out(
    model = model,
    corrupted_tokens = corrupt_dataset, # corrupted input sentences, with S2 swapped with IO. Are we patching with corrupted tokens?
    clean_cache = clean_cache, # clean cache, so clean activations
    patching_metric = patching_metric
)

In [ ]:
imshow(
    act_patch_attn_out,
    labels={"x": "Position", "y": "Layer"},
    x=labels,
    title="attn_out Activation Patching",
    width=600
)

In [ ]:
## patch each head output specifically
act_patch_attn_head_out_all_pos = patching.get_act_patch_attn_head_out_all_pos(
    model,
    corrupt_dataset,
    clean_cache,
    patching_metric
)

In [ ]:
imshow(
    act_patch_attn_head_out_all_pos,
    labels={"y": "Layer", "x": "Head"},
    title="attn_head_out Activation Patching (All Pos)",
    width=600
)

In [ ]:
"""
Rather than just patching on head output (like the previous one), it patches on:

Output (this is equivalent to patching the value the head writes to the residual stream)
Querys (i.e. the patching the query vectors, without changing the key or value vectors)
Keys
Values
Patterns (i.e. the attention patterns).
"""

act_patch_attn_head_all_pos_every = patching.get_act_patch_attn_head_all_pos_every(
    model,
    corrupt_dataset,
    clean_cache,
    patching_metric
)

In [ ]:
imshow(
    act_patch_attn_head_all_pos_every,
    facet_col=0,
    facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
    title="Activation Patching Per Head (All Pos)",
    labels={"x": "Head", "y": "Layer"},
)